# 03c Historical behavior features

Optional, for after the session. Notebook 03 gave the global LightGBM the history of prices and promotions (`lgbm_hist_drivers`). Does knowing more about how a SKU has behaved improve it further?

The control is `lgbm_hist_drivers`: the 02 baseline (LightGBM parameters, L2 objective, per-series standardized target, lags, rolling means and calendar features), department, class and subclass as categories, and the historical promotion and price drivers of notebook 03, section 3. Each experiment adds one coherent feature group to it and changes nothing else. Every model uses only information observed on the cutoff, so all of them are usable without future price or promotion plans.

| model | adds |
|---|---|
| `lgbm_hist_distribution` | demand volatility, momentum and zero frequency |
| `lgbm_hist_shape` | the distribution group plus rolling skewness and kurtosis |
| `lgbm_hist_promo_response` | promotion events, the length of the last one, and historical response to promotions |
| `lgbm_hist_price_response` | how often the price changes, and an observational price-response estimate |
| `lgbm_hist_static` | origin-level summaries of demand, zeros, promotions and price |
| `lgbm_hist_dept_agg`, `_class_agg`, `_dept_class_agg`, `_subclass_agg` | demand history of the product's department, class or subclass |
| `lgbm_hist_combined` | the groups that helped on the development origins |

Runs in about ten minutes.

In [1]:
import os
import sys
import warnings

if "google.colab" in sys.modules:
    !git clone -q https://github.com/tackes/time-series-forecasting-bootcamp
    %cd time-series-forecasting-bootcamp
    %pip install -q statsforecast==2.1.1 mlforecast==1.1.0 utilsforecast==0.2.17 lightgbm==4.7.0 catboost==1.2.10
elif not os.path.exists("shared"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

In [2]:
import operator
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, RollingMax, Combine
from mlforecast.target_transforms import LocalStandardScaler

from shared.prepare_data import load_panel, series_segments, H, FREQ, ALL_CUTOFFS, DEV_CUTOFFS, TEST_CUTOFF
from shared.features import LAGS, DATE_FEATURES, LGB_PARAMS, CATEGORICAL, HISTORICAL, add_drivers, item_attributes, historical_drivers, historical_future
from shared.evaluation import score
from shared.leaderboard import to_long, save_forecasts, load_forecasts, leaderboard, breakdown

panel = load_panel()
panel = add_drivers(panel)
actuals = panel[["unique_id", "ds", "y"]]

## 1. What is known when

The forecast is 14 weeks long, and the features fall into three kinds:

- **Recursive, from the target.** MLForecast lag transforms such as `RollingStd` on lag 1. In the forecast weeks they are recomputed from the model's own forecasts, like the control's lags and rolling means. They are computed on the standardized target, so they are already scale-free.
- **History as of 14 weeks earlier.** For a row in week *t*, a statistic of the weeks up to *t* - 14. For every forecast week that is data up to the cutoff, so it goes in `X_df` with no assumption about the future. The control's promotion and price drivers are of this kind (the regular price is carried forward from the cutoff), and so are the new promotion and price groups, so no forecast depends on promotions or prices inside the horizon.
- **Static per origin.** Summaries of each SKU's training history at the cutoff, recomputed at every origin and constant over the horizon.

`past()` builds the second kind. Every rolling window ends 14 weeks before the row's week, within each series.

In [3]:
def past(values, window, how, min_periods=1):
    """Rolling statistic per series over the `window` weeks ending H weeks before each row's week."""
    return values.groupby(panel.unique_id).transform(lambda s: s.shift(H).rolling(window, min_periods=min_periods).agg(how))


def weeks_since(event):
    """Weeks from the event's last occurrence to H weeks before each row's week (capped at 52)."""
    last = panel.ds.where(event).groupby(panel.unique_id).ffill().groupby(panel.unique_id).shift(H)
    return ((panel.ds - last).dt.days / 7 - H).clip(upper=52).fillna(52)


y = panel.y
features = historical_drivers(panel)      # the control's columns; the groups below are added to this frame

## 2. Demand distribution and volatility

A moving average says how much a SKU sells. These say how it sells: how much weeks vary, how often they are zero, and whether recent demand runs above or below the longer average. A global model can use them to tell a stable product from an intermittent or spiky one with the same average.

Recursive: standard deviation over 4, 13 and 26 weeks, the 13-week maximum, and momentum as the 4-week minus the 26-week mean. History as of 14 weeks earlier: zero share over 13 and 26 weeks, coefficient of variation over 26 weeks (at least 8 weeks, positive mean) and weeks since the last sale. Zero share and CV need the original units, which the standardized target no longer has.

The shape extension adds 26-week skewness and kurtosis (at least 8 weeks).

In [4]:
distribution_transforms = {1: [
    RollingMean(window_size=4), RollingMean(window_size=13), RollingMean(window_size=26),
    RollingStd(window_size=4), RollingStd(window_size=13), RollingStd(window_size=26), RollingMax(window_size=13),
    Combine(RollingMean(window_size=4), RollingMean(window_size=26), operator.sub),
]}

is_zero = (y == 0).astype(float)
features["zero_share_13"] = past(is_zero, 13, "mean")
features["zero_share_26"] = past(is_zero, 26, "mean")
features["cv_26"] = past(y, 26, "std", 8) / past(y, 26, "mean", 8).where(lambda m: m > 0)
features["weeks_since_sale"] = weeks_since(y > 0)
features["skew_26"] = past(y, 26, "skew", 8)
features["kurt_26"] = past(y, 26, "kurt", 8)

## 3. Promotion response

The control already knows the promotion flag 14 weeks back, the share of promotion weeks and the weeks since the last promotion. This group adds events and response. A promotion **event** is a run of consecutive promotion weeks, so a four-week promotion is one event and four promotion weeks.

- Events in the last 26 weeks and the length of the most recent event.
- Over the last 52 weeks: mean sales in promotion weeks over mean sales in other weeks (at least 3 promotion and 8 other weeks), and mean sales in the first week after a promotion over the same reference (at least 2 such weeks). Missing otherwise; LightGBM handles missing values.

All as of 14 weeks earlier. These are associations: promotions are chosen, not assigned at random.

In [5]:
promo = panel.promo.astype(float)
previous = panel.groupby("unique_id").promo.shift(1).fillna(0)
onset = ((panel.promo == 1) & (previous == 0)).astype(float)
after = ((panel.promo == 0) & (previous == 1)).astype(float)
run_length = promo.groupby((panel.promo != panel.groupby("unique_id").promo.shift()).cumsum()).cumsum()

features["promo_events_26"] = past(onset, 26, "sum")
features["last_event_weeks"] = run_length.where(promo == 1).groupby(panel.unique_id).ffill().groupby(panel.unique_id).shift(H).fillna(0)

promo_n, base_n, after_n = past(promo, 52, "sum"), past(1 - promo, 52, "sum"), past(after, 52, "sum")
base_mean = past(y * (1 - promo), 52, "sum") / base_n
features["promo_uplift_52"] = (past(y * promo, 52, "sum") / promo_n / base_mean).where((promo_n >= 3) & (base_n >= 8) & (base_mean > 0)).clip(upper=10)
features["post_promo_ratio_52"] = (past(y * after, 52, "sum") / after_n / base_mean).where((after_n >= 2) & (base_n >= 8) & (base_mean > 0)).clip(upper=10)

## 4. Price response

`rel_price` is the realized average selling price over the regular price. It is partly an outcome: a week with no sales gets the regular price (notebook 03, section 6), and 37% of weeks with sales sell above the regular price, which points to mix effects in the average. So, as in the control, it is used only as history. The control already has relative price 14 weeks back, mean discount and the mean and standard deviation of relative price. This group adds:

- the share of weeks in the last 26 where relative price changed by more than 2%;
- **price response**: the slope of log sales on log relative price over the last 52 weeks, using only weeks with sales. It needs 20 such weeks and some price variation, and is shrunk toward zero by `n var / (n var + 1)`, so a slope from little variation stays close to zero.

The slope is an observational association, not an elasticity. Across SKUs it is negative for 89% (median about -1.9), and estimates from the first and second half of each history correlate at only 0.38.

In [6]:
rel = panel.rel_price
features["price_change_share_26"] = past((rel.groupby(panel.unique_id).diff().abs() > 0.02).astype(float), 26, "mean")

selling = (y > 0) & (rel > 0)
log_price, log_sales = np.log(rel.where(selling)), np.log(y.where(selling))
n = past(selling.astype(float), 52, "sum")
mean_p, mean_s = past(log_price, 52, "mean"), past(log_sales, 52, "mean")
var_p = past(log_price ** 2, 52, "mean") - mean_p ** 2
cov_ps = past(log_price * log_sales, 52, "mean") - mean_p * mean_s
slope = (cov_ps / var_p).where((n >= 20) & (var_p > 1e-3))
features["price_response_52"] = slope * n * var_p / (n * var_p + 1)

## 5. Static summaries per origin

Similar to the static features in the Day 2 neural models, but recomputed from the training rows at each cutoff, never from the full history. They are constant for a series, so they go in `static_features` and MLForecast repeats them for the forecast weeks.

In [7]:
def origin_summary(history):
    g = history.groupby("unique_id")
    mean = g.y.mean()
    return pd.DataFrame({
        "log_mean_sales": np.log1p(mean),
        "cv_sales": g.y.std() / mean.where(mean > 0),
        "share_zero_weeks": (history.y == 0).groupby(history.unique_id).mean(),
        "share_promo_weeks": g.promo.mean(),
        "promo_events_per_year": onset[history.index].groupby(history.unique_id).sum() / g.size() * 52,
        "rel_price_std": g.rel_price.std(),
    }).reset_index()


static_cols = ["log_mean_sales", "cv_sales", "share_zero_weeks", "share_promo_weeks", "promo_events_per_year", "rel_price_std"]

## 6. Hierarchy demand

The categorical attributes tell the model which group a product belongs to. These describe how that group's demand has moved: weekly sales summed over the products of the 2,000-series sample in the same department, class or subclass. Membership is a fixed item attribute and the sample is fixed, so no future product activity decides who is in a group. A group total grows when a product starts selling, which is why the features are relative:

- Totals 14, 18, 26 and 52 weeks back, and 4-, 13- and 26-week means ending 14 weeks back, each divided by the 52-week mean ending 14 weeks back.
- 13-week coefficient of variation, momentum (4-week over 26-week mean), change from 18 to 14 weeks back, and the group's 13-week promotion share, all ending 14 weeks back.

In [8]:
cat_cols = CATEGORICAL
attributes = item_attributes(panel)
membership = panel[["unique_id", "ds"]].merge(attributes, on="unique_id", how="left")

aggregate_cols = {}
for level in cat_cols:
    weekly = panel.assign(group=membership[level]).groupby(["group", "ds"]).agg(total=("y", "sum"), promo_share=("promo", "mean")).reset_index()
    by_group = weekly.groupby("group")
    lagged = by_group.total.shift(H)
    rolled = lambda values, window, how: values.groupby(weekly.group).transform(lambda s: s.rolling(window, min_periods=1).agg(how))
    reference = lagged.groupby(weekly.group).transform(lambda s: s.rolling(52, min_periods=26).mean())

    group_features = pd.DataFrame({
        "group": weekly.group, "ds": weekly.ds,
        f"{level}_lag14": lagged / reference,
        f"{level}_lag18": by_group.total.shift(18) / reference,
        f"{level}_lag26": by_group.total.shift(26) / reference,
        f"{level}_lag52": by_group.total.shift(52) / reference,
        f"{level}_mean4": rolled(lagged, 4, "mean") / reference,
        f"{level}_mean13": rolled(lagged, 13, "mean") / reference,
        f"{level}_mean26": rolled(lagged, 26, "mean") / reference,
        f"{level}_cv13": rolled(lagged, 13, "std") / rolled(lagged, 13, "mean"),
        f"{level}_momentum": rolled(lagged, 4, "mean") / rolled(lagged, 26, "mean"),
        f"{level}_change": lagged / by_group.total.shift(18) - 1,
        f"{level}_promo_share13": rolled(by_group.promo_share.shift(H), 13, "mean"),
    }).replace([np.inf, -np.inf], np.nan)
    aggregate_cols[level] = list(group_features.columns[2:])
    features = features.assign(group=membership[level].values).merge(group_features, on=["group", "ds"], how="left").drop(columns="group")

features[["unique_id", "ds"] + aggregate_cols["department"][:4]].dropna().head(3)

,unique_id,ds,department_lag14,department_lag18,department_lag26,department_lag52
52,10006_S1,2024-07-03,1.149846,0.907158,1.067984,0.742815
53,10006_S1,2024-07-10,0.895604,1.373186,1.222724,0.935201
54,10006_S1,2024-07-17,0.958766,1.232249,1.006687,0.741571


## 7. The experiments

Each entry lists the dynamic columns (in the training frame and `X_df`), the lag transforms and the static columns. Every model gets the three categorical attributes, the control's historical drivers and the control's LightGBM setup.

In [9]:
control_transforms = {1: [RollingMean(window_size=4), RollingMean(window_size=13), RollingMean(window_size=26)]}
distribution_cols = ["zero_share_13", "zero_share_26", "cv_26", "weeks_since_sale"]
promo_cols = ["promo_events_26", "last_event_weeks", "promo_uplift_52", "post_promo_ratio_52"]
price_cols = ["price_change_share_26", "price_response_52"]


experiments = {
    "lgbm_hist_drivers": dict(cols=HISTORICAL, transforms=control_transforms, statics=[]),
    "lgbm_hist_distribution": dict(cols=HISTORICAL + distribution_cols, transforms=distribution_transforms, statics=[]),
    "lgbm_hist_shape": dict(cols=HISTORICAL + distribution_cols + ["skew_26", "kurt_26"], transforms=distribution_transforms, statics=[]),
    "lgbm_hist_promo_response": dict(cols=HISTORICAL + promo_cols, transforms=control_transforms, statics=[]),
    "lgbm_hist_price_response": dict(cols=HISTORICAL + price_cols, transforms=control_transforms, statics=[]),
    "lgbm_hist_static": dict(cols=HISTORICAL, transforms=control_transforms, statics=static_cols),
    "lgbm_hist_dept_agg": dict(cols=HISTORICAL + aggregate_cols["department"], transforms=control_transforms, statics=[]),
    "lgbm_hist_class_agg": dict(cols=HISTORICAL + aggregate_cols["class"], transforms=control_transforms, statics=[]),
    "lgbm_hist_dept_class_agg": dict(cols=HISTORICAL + aggregate_cols["department"] + aggregate_cols["class"], transforms=control_transforms, statics=[]),
    "lgbm_hist_subclass_agg": dict(cols=HISTORICAL + aggregate_cols["subclass"], transforms=control_transforms, statics=[]),
}

### One experiment at the final cutoff

The training frame for one experiment: sales up to the cutoff, the experiment's dynamic columns, the attributes and its static summaries. `X_df` has the same dynamic columns for the forecast weeks, with the regular price carried forward from the cutoff (`historical_future`, as in notebook 03).

In [10]:
name, spec = "lgbm_hist_price_response", experiments["lgbm_hist_price_response"]
summary = origin_summary(panel[panel.ds <= TEST_CUTOFF])
train = (actuals[actuals.ds <= TEST_CUTOFF]
         .merge(features[["unique_id", "ds"] + spec["cols"]], on=["unique_id", "ds"])
         .merge(attributes, on="unique_id")
         .merge(summary[["unique_id"] + spec["statics"]], on="unique_id")
         .astype({col: "category" for col in cat_cols}))
X_df = historical_future(features, TEST_CUTOFF)[["unique_id", "ds"] + spec["cols"]]

mlf = MLForecast(
    models={name: lgb.LGBMRegressor(**LGB_PARAMS)},
    freq=FREQ,
    lags=LAGS,
    lag_transforms=spec["transforms"],
    date_features=DATE_FEATURES,
    target_transforms=[LocalStandardScaler()],
    num_threads=1,
)
mlf.fit(train, static_features=cat_cols + spec["statics"])
display(X_df.head(3))
mlf.predict(H, X_df=X_df).head(3)

,unique_id,ds,promo_lag14,promo_share_26,weeks_since_promo_14,rel_price_lag14,discount_26,rel_price_mean_13,rel_price_std_13,regular_price,price_change_share_26,price_response_52
93,10006_S1,2025-04-16,0.0,0.115385,5.0,0.994759,0.041900,0.957582,0.091924,2.95,0.346154,-0.833668
94,10006_S1,2025-04-23,0.0,0.115385,6.0,0.997149,0.041141,0.957822,0.092032,2.95,0.307692,-0.830149
95,10006_S1,2025-04-30,0.0,0.115385,7.0,0.974827,0.041061,0.972971,0.074482,2.95,0.346154,-0.830017


,unique_id,ds,lgbm_hist_price_response
0,10006_S1,2025-04-16,1751.522704
1,10006_S1,2025-04-23,1830.182975
2,10006_S1,2025-04-30,1892.426408


### Every experiment at every origin

The same code for each experiment and cutoff. The control is refit here only to time it with the same code; its scores use the saved notebook 03 forecasts, which this code reproduces.

In [11]:
behavior_forecasts, fit_seconds = [], {}
for cutoff in ALL_CUTOFFS:
    history = actuals[actuals.ds <= cutoff]
    horizon = historical_future(features, cutoff)
    summary = origin_summary(panel[panel.ds <= cutoff])
    for name, spec in experiments.items():
        train = (history.merge(features[["unique_id", "ds"] + spec["cols"]], on=["unique_id", "ds"])
                 .merge(attributes, on="unique_id")
                 .merge(summary[["unique_id"] + spec["statics"]], on="unique_id")
                 .astype({col: "category" for col in cat_cols}))
        start = time.perf_counter()
        mlf = MLForecast(
            models={name: lgb.LGBMRegressor(**LGB_PARAMS)},
            freq=FREQ, lags=LAGS, lag_transforms=spec["transforms"], date_features=DATE_FEATURES,
            target_transforms=[LocalStandardScaler()], num_threads=1,
        )
        mlf.fit(train, static_features=cat_cols + spec["statics"])
        preds = mlf.predict(H, X_df=horizon[["unique_id", "ds"] + spec["cols"]])
        fit_seconds[name] = fit_seconds.get(name, 0) + time.perf_counter() - start
        behavior_forecasts.append(to_long(preds, cutoff, "03c_behavior_features", "historical_drivers"))

behavior_forecasts = pd.concat(behavior_forecasts)

The attributes are still treated as categories, and the new columns as numbers (last model fitted, final origin):

In [12]:
infos = mlf.models_["lgbm_hist_subclass_agg"].booster_.dump_model()["feature_infos"]
pd.Series({f: "categorical" if i["values"] else "numeric" for f, i in infos.items()}).value_counts()

numeric        33
categorical     3
Name: count, dtype: int64

## 8. Results

Each model against the control on identical rows. Fit time is fit plus 14-week forecast, summed over the ten origins.

In [13]:
control = load_forecasts(["03_exogenous"]).query("model == 'lgbm_hist_drivers'")
new_models = [name for name in experiments if name != "lgbm_hist_drivers"]
candidates = pd.concat([control, behavior_forecasts[behavior_forecasts.model.isin(new_models)]])

columns = ["model", "wape", "bias", "mean_rmsse", "median_rmsse", "apc", "coverage"]
dev_board = leaderboard(candidates, actuals, scope="dev").set_index("model")
test_board = leaderboard(candidates, actuals, scope="test").set_index("model")
table = pd.concat({"development": dev_board[columns[1:]], "final 14 weeks": test_board[columns[1:3]]}, axis=1)
table[("fit", "seconds")] = pd.Series(fit_seconds)
table.sort_values(("development", "wape")).round(3)

development                                        \
                                wape   bias mean_rmsse median_rmsse    apc   
model                                                                        
lgbm_hist_subclass_agg         0.411  0.106      1.806        0.879  0.146   
lgbm_hist_price_response       0.412  0.086      1.829        0.886  0.150   
lgbm_hist_promo_response       0.413  0.084      1.834        0.884  0.150   
lgbm_hist_static               0.414  0.090      1.847        0.879  0.146   
lgbm_hist_drivers              0.416  0.091      1.837        0.884  0.149   
lgbm_hist_dept_class_agg       0.416  0.113      1.819        0.887  0.147   
lgbm_hist_class_agg            0.418  0.115      1.825        0.883  0.147   
lgbm_hist_dept_agg             0.420  0.116      1.837        0.888  0.148   
lgbm_hist_shape                0.420  0.090      1.849        0.886  0.147   
lgbm_hist_distribution         0.422  0.094      1.845        0.889  0.149   

                                  final 14 weeks            fit  
                         coverage           wape   bias seconds  
model                                                            
lgbm_hist_subclass_agg        1.0          0.397  0.123  17.421  
lgbm_hist_price_response      1.0          0.395  0.104  16.406  
lgbm_hist_promo_response      1.0          0.393  0.100  17.044  
lgbm_hist_static              1.0          0.403  0.121  17.525  
lgbm_hist_drivers             1.0          0.396  0.107  16.325  
lgbm_hist_dept_class_agg      1.0          0.398  0.129  19.551  
lgbm_hist_class_agg           1.0          0.394  0.119  17.314  
lgbm_hist_dept_agg            1.0          0.404  0.126  17.356  
lgbm_hist_shape               1.0          0.408  0.114  17.989  
lgbm_hist_distribution        1.0          0.407  0.114  18.391

Change in WAPE points against the control at every origin; `beats_dev` counts the seven development origins where the model is better.

In [14]:
origins = list(DEV_CUTOFFS) + [TEST_CUTOFF]
by_origin = pd.concat([score(candidates[candidates.cutoff == c], actuals, rmsse=False).assign(cutoff=c) for c in origins])
origin_wape = by_origin.pivot(index="model", columns="cutoff", values="wape")
vs_control = (origin_wape - origin_wape.loc["lgbm_hist_drivers"]) * 100
vs_control.drop(index="lgbm_hist_drivers").round(2).assign(beats_dev=(vs_control[list(DEV_CUTOFFS)] < 0).sum(axis=1))

cutoff,2024-07-17 00:00:00,2024-08-14 00:00:00,2024-09-11 00:00:00,2024-10-09 00:00:00,2024-11-06 00:00:00,2024-12-04 00:00:00,2025-01-01 00:00:00,2025-04-09 00:00:00,beats_dev
model,,,,,,,,,
lgbm_hist_class_agg,-0.23,-0.23,-0.93,-0.34,0.55,1.91,1.03,-0.25,4
lgbm_hist_dept_agg,-0.06,0.65,-0.13,0.19,0.04,1.68,0.89,0.78,2
lgbm_hist_dept_class_agg,0.12,-0.37,-0.17,-1.08,-0.46,1.30,0.93,0.22,4
lgbm_hist_distribution,0.35,0.73,0.36,1.65,0.25,1.14,0.17,1.05,0
lgbm_hist_price_response,-0.86,-0.59,-0.58,0.21,-0.17,0.04,-0.42,-0.07,5
lgbm_hist_promo_response,-1.36,-0.16,-0.51,-0.15,0.22,0.66,-0.41,-0.36,5
lgbm_hist_shape,-0.52,0.00,0.16,1.89,0.61,1.10,-0.15,1.14,2
lgbm_hist_static,-1.23,0.70,-0.54,-0.41,0.41,0.29,-0.43,0.66,4
lgbm_hist_subclass_agg,-1.17,0.05,-1.13,-0.59,-1.29,0.53,0.21,0.03,4


### By product characteristics

Segments are recomputed at each origin from history up to that cutoff: volume and intermittency with `series_segments(panel, cutoff)`, and promotion frequency as the share of promotion weeks (rarely up to 5%, occasionally up to 25%, heavily above). Development origins; WAPE change in points against the control.

In [15]:
def origin_segments(cutoff):
    segments = series_segments(panel, cutoff)
    share = panel[panel.ds <= cutoff].groupby("unique_id").promo.mean()
    segments["promotion"] = pd.cut(share, [-1, 0.05, 0.25, 1], labels=["rarely", "occasionally", "heavily"]).astype(str)
    return segments.assign(cutoff=cutoff).rename_axis("unique_id").reset_index()


segment_rows = candidates[candidates.cutoff.isin(DEV_CUTOFFS)].merge(pd.concat([origin_segments(c) for c in DEV_CUTOFFS]), on=["unique_id", "cutoff"])
segment_change = {}
for segment, order in [("volume", ["low", "mid", "high"]), ("intermittency", ["regular", "some_zeros", "zero_heavy"]), ("promotion", ["rarely", "occasionally", "heavily"])]:
    wape = score(segment_rows, actuals, by=[segment], rmsse=False).pivot(index="model", columns=segment, values="wape")[order]
    segment_change[segment] = (wape - wape.loc["lgbm_hist_drivers"]) * 100
pd.concat(segment_change, axis=1).drop(index="lgbm_hist_drivers").round(2)

volume             intermittency             \
                            low   mid  high       regular some_zeros   
model                                                                  
lgbm_hist_class_agg       -0.37 -0.44  0.33          0.44       0.01   
lgbm_hist_dept_agg        -0.09  0.04  0.51          0.68      -0.03   
lgbm_hist_dept_class_agg  -0.50 -0.50  0.09          0.34      -0.02   
lgbm_hist_distribution     0.56  0.64  0.68          0.47       0.45   
lgbm_hist_price_response  -0.32 -0.31 -0.33         -0.31      -0.79   
lgbm_hist_promo_response  -0.18 -0.10 -0.23         -0.30      -0.32   
lgbm_hist_shape            0.48  0.80  0.44          0.16       0.38   
lgbm_hist_static          -2.36 -0.30 -0.12         -0.57      -0.54   
lgbm_hist_subclass_agg    -0.58 -0.93 -0.44         -0.37      -0.44   

                                    promotion                       
                         zero_heavy    rarely occasionally heavily  
model                                                               
lgbm_hist_class_agg           -2.90      0.03        -0.16    0.50  
lgbm_hist_dept_agg            -0.90      0.07         0.48    0.60  
lgbm_hist_dept_class_agg      -9.70     -0.12        -0.56    0.31  
lgbm_hist_distribution        10.16      0.15         1.73    0.45  
lgbm_hist_price_response       4.54     -0.07         0.45   -0.70  
lgbm_hist_promo_response       3.71      0.14         0.48   -0.61  
lgbm_hist_shape               12.25      0.16         1.76    0.09  
lgbm_hist_static              18.24      0.23         1.86   -1.05  
lgbm_hist_subclass_agg        -4.99     -0.18        -1.06   -0.37

## 9. Combined model, chosen on development evidence

A group goes into the combined model if it lowers development WAPE by at least 0.2 points and beats the control at five or more of the seven development origins. Within a family of overlapping variants (distribution and shape; the four aggregate variants) only the one with the lowest development WAPE is kept. The final window plays no part in the choice.

In [16]:
qualifies = ((dev_board.wape["lgbm_hist_drivers"] - dev_board.wape) >= 0.002) & (vs_control[list(DEV_CUTOFFS)].lt(0).sum(axis=1) >= 5)
families = [["distribution", "shape"], ["promo_response"], ["price_response"], ["static"], ["dept_agg", "class_agg", "dept_class_agg", "subclass_agg"]]
chosen = []
for family in families:
    passing = [f"lgbm_hist_{group}" for group in family if qualifies[f"lgbm_hist_{group}"]]
    chosen += [min(passing, key=dev_board.wape.get)] if passing else []
chosen

['lgbm_hist_promo_response', 'lgbm_hist_price_response']

In [17]:
added = [col for name in chosen for col in experiments[name]["cols"] if col not in HISTORICAL]
combined = dict(
    cols=HISTORICAL + sorted(set(added)),
    transforms=distribution_transforms if {"lgbm_hist_distribution", "lgbm_hist_shape"} & set(chosen) else control_transforms,
    statics=sorted({col for name in chosen for col in experiments[name]["statics"]}),
)

combined_forecasts = []
for cutoff in ALL_CUTOFFS:
    summary = origin_summary(panel[panel.ds <= cutoff])
    train = (actuals[actuals.ds <= cutoff]
             .merge(features[["unique_id", "ds"] + combined["cols"]], on=["unique_id", "ds"])
             .merge(attributes, on="unique_id")
             .merge(summary[["unique_id"] + combined["statics"]], on="unique_id")
             .astype({col: "category" for col in cat_cols}))
    start = time.perf_counter()
    mlf = MLForecast(
        models={"lgbm_hist_combined": lgb.LGBMRegressor(**LGB_PARAMS)},
        freq=FREQ, lags=LAGS, lag_transforms=combined["transforms"], date_features=DATE_FEATURES,
        target_transforms=[LocalStandardScaler()], num_threads=1,
    )
    mlf.fit(train, static_features=cat_cols + combined["statics"])
    preds = mlf.predict(H, X_df=historical_future(features, cutoff)[["unique_id", "ds"] + combined["cols"]])
    fit_seconds["lgbm_hist_combined"] = fit_seconds.get("lgbm_hist_combined", 0) + time.perf_counter() - start
    combined_forecasts.append(to_long(preds, cutoff, "03c_behavior_features", "historical_drivers"))
combined_forecasts = pd.concat(combined_forecasts)

candidates = pd.concat([candidates, combined_forecasts])
best_group = dev_board.drop(index="lgbm_hist_drivers").wape.idxmin()
compare = candidates[candidates.model.isin(["lgbm_hist_drivers", best_group, "lgbm_hist_combined"])]
pd.concat({"development": leaderboard(compare, actuals, scope="dev").set_index("model")[columns[1:]],
           "final 14 weeks": leaderboard(compare, actuals, scope="test").set_index("model")[columns[1:3]]}, axis=1).round(3)

development                                        \
                              wape   bias mean_rmsse median_rmsse    apc   
model                                                                      
lgbm_hist_subclass_agg       0.411  0.106      1.806        0.879  0.146   
lgbm_hist_combined           0.411  0.082      1.831        0.882  0.148   
lgbm_hist_drivers            0.416  0.091      1.837        0.884  0.149   

                                final 14 weeks         
                       coverage           wape   bias  
model                                                  
lgbm_hist_subclass_agg      1.0          0.397  0.123  
lgbm_hist_combined          1.0          0.395  0.100  
lgbm_hist_drivers           1.0          0.396  0.107

### By horizon

Development WAPE by forecast week for the control, the best single group and the combined model.

In [18]:
breakdown(compare, actuals, "horizon", scope="dev").pivot(index="horizon", columns="model", values="wape").round(3)

model,lgbm_hist_combined,lgbm_hist_drivers,lgbm_hist_subclass_agg
horizon,,,
1,0.288,0.292,0.286
2,0.336,0.337,0.328
3,0.375,0.378,0.368
4,0.381,0.384,0.383
5,0.392,0.398,0.388
6,0.427,0.433,0.422
7,0.426,0.432,0.428
8,0.435,0.438,0.434
9,0.429,0.434,0.429


## 10. Reading the result

Control (`lgbm_hist_drivers`): development WAPE 0.416, final window 0.396.

- **No group adds much.** The best on development are the subclass demand history (0.411, better at four of seven origins), price response (0.412, five of seven), promotion response (0.413, five of seven) and the static summaries (0.414, four of seven). Distribution and shape features are worse (0.422 and 0.420).
- **Combined.** Promotion response and price response pass the rule; together they reach 0.411 on development, at about the same level across all horizons, and 0.395 in the final window against 0.396.
- **Final window.** Every group lies within about a point of the control; distribution, shape, static and department demand are worse there.
- **Segments.** Zero-heavy products swing by several points either way, because their WAPE rests on very few sales.
- **RMSSE and APC barely move** for any group.

Once the historical price and promotion drivers are in, further summaries of past behavior add half a point at most. The gap to the plan models of notebook 03 (0.311) is about knowing the forecast weeks, which history cannot supply.

In [19]:
save_forecasts(candidates[candidates.model != "lgbm_hist_drivers"], "03c_behavior_features")